# Reading chess books for MeChess

Reads public-domain chess books (and, optionally, PDFs **you own**) and produces training material for the style and move-classification work:
game lines pulled from the text (algebraic *and* descriptive notation), concept mentions in the prose, and paragraphs that pair a concept with a game line.

**Run on Kaggle**
1. *Settings -> Internet -> On* (Kaggle asks for phone verification once). Part A and B need no GPU; Part D is faster with a GPU (*Accelerator*).
2. Put your repository URL in `REPO_URL` below and *Run all*. Results appear under `/kaggle/working/` (download `output.zip`).
3. Parts C (your own PDFs) and D (NLP) are switched off by default (`RUN_PDF`, `RUN_NLP`).

**Rules this notebook follows**
- Only the public-domain books listed in `chessme/books/sources.json` are downloaded. Modern books are copyrighted: use Part C only for books you own, keep that notebook **private**, and never publish or commit the extracted text.
- Requests are one at a time with pauses and retries (Project Gutenberg and the Internet Archive are volunteer-run).

In [ ]:
import os, subprocess, sys, json, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
RUN_PDF = False                                        # Part C: your own PDFs (private notebook only)
RUN_NLP = False                                        # Part D: paragraph embeddings and clusters
OUT = pathlib.Path("/kaggle/working/books") if os.path.exists("/kaggle") else pathlib.Path("books_out")

if not os.path.exists("chessme"):                      # on Kaggle: clone the repository (needs Internet on)
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "MeChess"], check=True)
    os.chdir("MeChess")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-chess", "pandas"], check=True)
sys.path.insert(0, os.getcwd())
OUT.mkdir(parents=True, exist_ok=True)
print("working in", os.getcwd(), "-> output", OUT)

## Part A. Download and read the public-domain books

In [ ]:
from chessme.books import fetch, text

sources = fetch.load_sources()
print(len(sources), "books:", ", ".join(b["id"] for b in sources))
result = fetch.run(OUT)                    # resumable: finished books are skipped, failures can be retried by running again
print(result)

## Part B. What did we get?

In [ ]:
import pandas as pd

rows = []
for f in sorted(OUT.glob("*.json")):
    d = json.loads(f.read_text())
    plies = sum(len(l["moves"]) for l in d["lines"])
    top = sorted(d["concepts"].items(), key=lambda kv: -kv[1])[:4]
    rows.append({"book": d["id"], "declared": d["declared_notation"], "paragraphs": d["paragraphs"], "lines": len(d["lines"]),
                 "plies": plies, "descriptive": d["notation"]["descriptive"], "algebraic": d["notation"]["algebraic"],
                 "paragraphs with concept and line": sum(d["concept_with_moves"].values()), "top concepts": ", ".join(f"{k}:{v}" for k, v in top)})
summary = pd.DataFrame(rows)
summary

In [ ]:
# A few extracted lines, to check by eye that the parser reads what the book says
import random
random.seed(0)
for f in sorted(OUT.glob("*.json"))[:4]:
    d = json.loads(f.read_text())
    for l in random.sample(d["lines"], min(2, len(d["lines"]))):
        print(f"{d['id']:32s} {l['notation']:11s}", " ".join(l["moves"][:14]))

In [ ]:
# Weak-supervision pairs: a game line from the initial position, with the concepts the prose mentions in the same paragraph
# and in the WINDOW paragraphs before and after it (books put the explanation and the moves in neighbouring paragraphs)
WINDOW = 2
pairs = []
for f in sorted(OUT.glob("*.json")):
    book = f.stem
    paras = text.paragraphs(text.strip_gutenberg((OUT / f"{book}.txt").read_text(errors="replace")))
    for i, p in enumerate(paras):
        if not text.START.search(p):
            continue
        lines = text.find_lines(p)
        if not lines:
            continue
        near = {}
        for j in range(max(0, i - WINDOW), min(len(paras), i + WINDOW + 1)):
            for k, n in text.concept_counts(paras[j]).items():
                near[k] = near.get(k, 0) + n
        if near:
            pairs.append({"book": book, "paragraph": i, "concepts": near, "moves": lines[0]["moves"],
                          "text": " ".join(paras[max(0, i - WINDOW): i + WINDOW + 1])[:2400]})
with open(OUT / "concept_line_pairs.jsonl", "w") as fh:
    for r in pairs:
        fh.write(json.dumps(r) + "\n")
print(len(pairs), "game lines paired with nearby concepts ->", OUT / "concept_line_pairs.jsonl")

## Part C. Your own PDFs (optional, private notebook only)

Books that are scanned images need OCR; books with a text layer do not. Chess **diagrams** are pictures: recognising them needs a separate model
(for example the MIT-licensed `fenify` or `Chess_diagram_to_FEN` projects); until then only lines that start from the initial position are read.
Upload your PDFs as a *private* Kaggle dataset; they appear under `/kaggle/input/<dataset>/`.

In [ ]:
if RUN_PDF:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pypdf"], check=True)
    from pypdf import PdfReader
    PDF_DIR = pathlib.Path("/kaggle/input")                     # your private dataset
    for pdf in sorted(PDF_DIR.rglob("*.pdf")):
        pages = [(p.extract_text() or "") for p in PdfReader(str(pdf)).pages]
        chars = sum(len(t) for t in pages)
        if chars < 200 * max(len(pages), 1):                    # no real text layer: it is a scan, OCR it
            print(pdf.name, "has no text layer; OCR needed: apt-get install tesseract-ocr, then pip install ocrmypdf")
            continue
        body = "\n\n".join(pages)
        res = text.analyse_book(body)
        (OUT / f"private_{pdf.stem}.txt").write_text(body)
        print(pdf.name, len(pages), "pages;", len(res["lines"]), "lines;", res["notation"])
else:
    print("Part C is off (RUN_PDF = False)")

## Part D. Which concepts does the prose talk about? (optional NLP)

A sentence-embedding model turns paragraphs into vectors; clustering them shows the topics the books actually discuss, to compare with our hand-written concept list
(`chessme/books/text.py`, `LEXICON`). Concepts that appear in clusters but not in the list are candidates to add. Works on CPU, faster on a GPU.

In [ ]:
if RUN_NLP:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "sentence-transformers", "scikit-learn"], check=True)
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer
    paras = []
    for f in sorted(OUT.glob("*.txt")):
        for p in text.paragraphs(text.strip_gutenberg(f.read_text(errors="replace"))):
            if 200 <= len(p) <= 1500:
                paras.append(p)
    print(len(paras), "paragraphs")
    emb = SentenceTransformer("all-MiniLM-L6-v2").encode(paras, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    k = 25
    labels = KMeans(n_clusters=k, n_init=3, random_state=0).fit_predict(emb)
    tf = TfidfVectorizer(stop_words="english", max_features=20000).fit(paras)
    terms = tf.get_feature_names_out()
    for c in range(k):
        idx = [i for i, l in enumerate(labels) if l == c]
        scores = tf.transform([paras[i] for i in idx]).sum(axis=0).A1
        print(f"cluster {c:2d} ({len(idx):4d} paragraphs):", ", ".join(terms[j] for j in scores.argsort()[::-1][:8]))
else:
    print("Part D is off (RUN_NLP = False)")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/output" if os.path.exists("/kaggle") else "output", "zip", OUT)
print("done: download output.zip (the extracted lines, concept counts and pairs; the .txt files are the public-domain texts)")